# Joint Distributions

`ProductDistribution` combines named, independent distributions into a single joint distribution. Components are accessed via `joint['name']`, which returns a `DistributionView` — a lightweight reference that remembers its parent. When multiple `DistributionView` instances from the same parent are passed to different workflow arguments, broadcasting samples them **jointly**, preserving correlation. This is the key mechanism for correlated broadcasting through workflows.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from probpipe import (
    Normal, MultivariateNormal, ProductDistribution,
    DistributionView, EmpiricalDistribution,
)
from probpipe.core.node import Workflow

## Creating a ProductDistribution

A `ProductDistribution` is constructed by passing named distributions as keyword arguments. The components are independent — sampling draws from each component separately, and `log_prob` is the sum of component log-probs. The flat event shape is the sum of component dimensions.

In [ ]:
joint = ProductDistribution(
    theta=Normal(loc=0.0, scale=1.0),
    sigma=Normal(loc=1.0, scale=0.5),
)
print(f"Joint: {joint}")
print(f"Components: {joint.component_names}")
print(f"Event shape: {joint.event_shape}")
print(f"Mean: {joint.mean()}")
print(f"Variance: {joint.variance()}")

## Sampling

`sample()` returns a flat array of shape `(n, total_dim)` where `total_dim` is the sum of all component dimensions. `sample_structured()` returns a dict of per-component arrays, which is often more convenient.

In [ ]:
# Flat sampling: shape (n, total_dim)
flat = joint.sample(sample_shape=(5,))
print(f"Flat sample shape: {flat.shape}")
print(f"Flat samples:\n{flat}")

# Structured sampling: dict of per-component arrays
structured = joint.sample_structured(sample_shape=(5,))
print(f"\nStructured keys: {list(structured.keys())}")
print(f"theta samples: {structured['theta']}")
print(f"sigma samples: {structured['sigma']}")

## Accessing Components

`joint['name']` returns a `DistributionView` — a lightweight reference that remembers its parent joint distribution. Views can be sampled and evaluated like any distribution.

In [ ]:
view_theta = joint['theta']
print(f"View: {view_theta}")
print(f"Event shape: {view_theta.event_shape}")
print(f"Mean: {float(view_theta.mean()):.3f}")
# Sample from the view
s = view_theta.sample(sample_shape=(5,))
print(f"Samples: {s}")

## bind() for Name Remapping

`joint.bind(a='theta', b='sigma')` creates a dict of `DistributionView` instances that maps workflow argument names (e.g. `a`, `b`) to component names (e.g. `theta`, `sigma`). This is convenient for passing joint components to a workflow whose parameter names differ from the component names.

In [ ]:
views = joint.bind(a='theta', b='sigma')
print(f"Bound views: {views}")

## Correlated Broadcasting

This is the key feature: when two `DistributionView` instances from the **same parent** are passed to different workflow arguments, the broadcasting machinery samples the parent once and distributes the component samples. This means the views are sampled **jointly**, preserving any shared randomness.

To demonstrate, we pass `joint['theta']` to *both* arguments of a difference function. Since `a` and `b` receive the same samples, `a - b` should always be exactly zero.

In [ ]:
def difference(a: jnp.ndarray, b: jnp.ndarray) -> jnp.ndarray:
    return a - b

w = Workflow(func=difference, n_broadcast_samples=500, broadcast_backend="loop", seed=0)

# Pass two views from the same joint — same component, same parent
result = w(a=joint['theta'], b=joint['theta'])
# Since a and b are the SAME component sampled jointly, a-b should always be 0
print(f"Mean of (theta - theta): {float(jnp.mean(result.samples)):.6f}")
print(f"Std of (theta - theta): {float(jnp.std(result.samples)):.6f}")
print("(Should be ~0 since they're the same samples)")

In [ ]:
# If we pass two independent Normals instead, a-b will have nonzero variance
result_indep = w(a=Normal(loc=0.0, scale=1.0), b=Normal(loc=0.0, scale=1.0))
print(f"Mean of (indep1 - indep2): {float(jnp.mean(result_indep.samples)):.3f}")
print(f"Std of (indep1 - indep2): {float(jnp.std(result_indep.samples)):.3f}")
print("(Nonzero std since these are independent)")

## Multivariate Components

`ProductDistribution` works with multivariate components too. The flat event shape is the sum of all component event dimensions.

In [ ]:
joint_mv = ProductDistribution(
    pos=MultivariateNormal(loc=jnp.zeros(2), cov=jnp.eye(2)),
    vel=MultivariateNormal(loc=jnp.array([1.0, 0.0]), cov=0.1 * jnp.eye(2)),
)
ss = joint_mv.sample_structured(sample_shape=(500,))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(ss['pos'][:, 0], ss['pos'][:, 1], alpha=0.3, s=5)
ax1.set_title("Position")
ax1.axis("equal")
ax2.scatter(ss['vel'][:, 0], ss['vel'][:, 1], alpha=0.3, s=5, color="orange")
ax2.set_title("Velocity")
ax2.axis("equal")
plt.tight_layout()
plt.show()

## from_distributions()

`ProductDistribution.from_distributions()` constructs a joint distribution from a list of named distributions. Each distribution must have a non-None `name` attribute, and names must be unique.

In [ ]:
theta = Normal(loc=0.0, scale=1.0, name="theta")
sigma = Normal(loc=1.0, scale=0.5, name="sigma")
joint2 = ProductDistribution.from_distributions([theta, sigma])
print(f"Joint: {joint2}")
print(f"Components: {joint2.component_names}")

## Pinning

`joint.pin(name=value)` fixes a component to a concrete value, returning a new joint distribution. The pinned component becomes a `PinnedComponent` that always returns the fixed value when sampled, while other components continue to vary normally.

In [ ]:
pinned = joint.pin(theta=jnp.array(2.0))
print(f"Pinned joint: {pinned}")

ss_pinned = pinned.sample_structured(sample_shape=(5,))
print(f"\nTheta samples (all pinned to 2.0): {ss_pinned['theta']}")
print(f"Sigma samples (still vary): {ss_pinned['sigma']}")

In [ ]:
def compute_product(theta: jnp.ndarray, sigma: jnp.ndarray) -> jnp.ndarray:
    return theta * sigma

w2 = Workflow(func=compute_product, n_broadcast_samples=200, broadcast_backend="loop", seed=1)

# Without pinning: both vary
result_free = w2(**joint.bind(theta='theta', sigma='sigma'))
print(f"Free mean: {float(jnp.mean(result_free.samples)):.3f}")

# With pinning: theta fixed at 2
result_pinned = w2(**pinned.bind(theta='theta', sigma='sigma'))
print(f"Pinned mean (theta=2, sigma~N(1,0.5)): {float(jnp.mean(result_pinned.samples)):.3f}")
print(f"Expected: ~{2 * 1.0:.1f}")

## Log-Probability

For a `ProductDistribution`, `log_prob` on the flat representation equals the sum of component log-probs, since the components are independent.

In [ ]:
x = jnp.array([0.5, 1.5])  # [theta_val, sigma_val]
lp_joint = joint.log_prob(x)
lp_sum = joint.components['theta'].log_prob(0.5) + joint.components['sigma'].log_prob(1.5)
print(f"Joint log_prob: {float(lp_joint):.4f}")
print(f"Sum of marginals: {float(lp_sum):.4f}")
print(f"Equal: {bool(jnp.isclose(lp_joint, lp_sum))}")

## Summary

Key takeaways:

- **ProductDistribution**: named independent components combined into a single joint distribution.
- **DistributionView**: lightweight component reference for use in broadcasting.
- **Broadcasting reconnection**: views from the same parent are sampled jointly, preserving shared randomness.
- **Pinning**: fix components to concrete values with `joint.pin(name=value)`.
- **log_prob** = sum of marginal log-probs (by independence).